# Build Spatial Analysis Masks
Create binary masks for radial or morphology-based attribution analysis. These masks are not dark matter ground-truth labels.

In [ ]:
# Setup Google Colab if running in Colab environment
import sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    project_path = '/content/drive/MyDrive/xai-dark-matter-localization'
    sys.path.insert(0, project_path)
except ImportError:
    project_path = '/'.join(os.getcwd().split('/')[:-1])
    sys.path.insert(0, project_path)

os.chdir(project_path)
print(f"✓ Working directory: {os.getcwd()}")

## 1. Import Libraries

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import h5py
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import ndimage
from skimage import morphology
from tqdm.auto import tqdm

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 4)

print("✓ All libraries imported successfully")

## 2. Load Preprocessed Data

In [ ]:
# Configuration
RESOLUTION = 512
DATA_DIR = Path('data/processed/TNG-DM-XAI-v1')
PREPROCESSED_DIR = DATA_DIR / 'preprocessed'
H5_FILE = PREPROCESSED_DIR / f'images_{RESOLUTION}_preprocessed.h5'

# Load preprocessed images from HDF5
print(f"Loading preprocessed images from {H5_FILE}...")
with h5py.File(H5_FILE, 'r') as f:
    X_train = f['X_train'][:]
    X_val = f['X_val'][:]
    X_test = f['X_test'][:]

print(f"✓ Train images: {X_train.shape}")
print(f"✓ Val images: {X_val.shape}")
print(f"✓ Test images: {X_test.shape}")

# Load metadata
df_train = pd.read_csv(PREPROCESSED_DIR / f'metadata_train_{RESOLUTION}.csv')
df_val = pd.read_csv(PREPROCESSED_DIR / f'metadata_val_{RESOLUTION}.csv')
df_test = pd.read_csv(PREPROCESSED_DIR / f'metadata_test_{RESOLUTION}.csv')

print(f"\n✓ Metadata loaded:")
print(f"  Train: {len(df_train)}")
print(f"  Val: {len(df_val)}")
print(f"  Test: {len(df_test)}")
print(f"\nTrain metadata sample:\n{df_train.head(2)}")

## 3. Thresholding Configuration

In [ ]:
# Masking strategy: Use Otsu's method to automatically find optimal threshold
from skimage.filters import threshold_otsu, threshold_li

# Configuration for mask generation
THRESHOLD_METHOD = 'otsu'  # 'otsu', 'li', or 'manual'
MANUAL_THRESHOLD = 0.6     # Only used if THRESHOLD_METHOD='manual'
APPLY_MORPHOLOGY = True    # Apply erosion/dilation to clean masks
MORPH_SIZE = 3             # Structuring element size

print("Masking Configuration:")
print(f"  Threshold method: {THRESHOLD_METHOD}")
print(f"  Apply morphology: {APPLY_MORPHOLOGY}")
print(f"  Morphology size: {MORPH_SIZE}")

# Test on first train image to visualize
test_img = X_train[0].squeeze()
print(f"\nTest image stats:")
print(f"  Min: {test_img.min():.3f}")
print(f"  Max: {test_img.max():.3f}")
print(f"  Mean: {test_img.mean():.3f}")
print(f"  Median: {np.median(test_img):.3f}")

# Calculate thresholds
otsu_thresh = threshold_otsu(test_img)
li_thresh = threshold_li(test_img)
print(f"\nAutomatic thresholds:")
print(f"  Otsu: {otsu_thresh:.3f}")
print(f"  Li: {li_thresh:.3f}")

## 4. Create Masks Function

In [ ]:
def create_masks(images, threshold_method='otsu', manual_threshold=0.6, 
                 apply_morphology=True, morph_size=3):
    """
    Create binary spatial analysis masks from image intensity patterns.
    
    Args:
        images: Input images array (N, H, W, 1) or (N, H, W)
        threshold_method: 'otsu', 'li', or 'manual'
        manual_threshold: Threshold value if using 'manual' method
        apply_morphology: Apply morphological operations
        morph_size: Size of structuring element
    
    Returns:
        masks: Binary masks array (N, H, W, 1)
    """
    N = images.shape[0]
    H, W = images.shape[1], images.shape[2]
    masks = np.zeros((N, H, W, 1), dtype=np.uint8)
    
    se = morphology.square(morph_size)  # Structuring element
    
    for i in tqdm(range(N), desc="Creating masks"):
        # Get single image and squeeze
        img = images[i].squeeze()
        
        # Determine threshold
        if threshold_method == 'otsu':
            thresh = threshold_otsu(img)
        elif threshold_method == 'li':
            thresh = threshold_li(img)
        else:  # manual
            thresh = manual_threshold
        
        # Create binary mask
        mask = (img > thresh).astype(np.uint8)
        
        # Apply morphological operations to clean mask
        if apply_morphology:
            mask = morphology.binary_opening(mask, se).astype(np.uint8)
        
        masks[i, :, :, 0] = mask
    
    return masks

# Create masks for all splits
print("Creating masks for train set...")
masks_train = create_masks(X_train, threshold_method=THRESHOLD_METHOD, 
                           apply_morphology=APPLY_MORPHOLOGY, 
                           morph_size=MORPH_SIZE)

print("Creating masks for validation set...")
masks_val = create_masks(X_val, threshold_method=THRESHOLD_METHOD, 
                         apply_morphology=APPLY_MORPHOLOGY, 
                         morph_size=MORPH_SIZE)

print("Creating masks for test set...")
masks_test = create_masks(X_test, threshold_method=THRESHOLD_METHOD, 
                          apply_morphology=APPLY_MORPHOLOGY, 
                          morph_size=MORPH_SIZE)

print(f"\n✓ Masks created successfully:")
print(f"  Train masks: {masks_train.shape}")
print(f"  Val masks: {masks_val.shape}")
print(f"  Test masks: {masks_test.shape}")

## 5. Analyze Masks

In [ ]:
# Analyze mask statistics
def analyze_masks(masks, name=""):
    """Calculate mask statistics."""
    coverage = masks.mean(axis=(1, 2, 3)) * 100  # Percentage of pixels in mask
    print(f"\n{name} Mask Statistics:")
    print(f"  Total masks: {len(masks)}")
    print(f"  Coverage (mean): {coverage.mean():.2f}%")
    print(f"  Coverage (min): {coverage.min():.2f}%")
    print(f"  Coverage (max): {coverage.max():.2f}%")
    print(f"  Coverage (std): {coverage.std():.2f}%")
    return coverage

train_coverage = analyze_masks(masks_train, "Train")
val_coverage = analyze_masks(masks_val, "Val")
test_coverage = analyze_masks(masks_test, "Test")

# Combined coverage stats
all_coverage = np.concatenate([train_coverage, val_coverage, test_coverage])
print(f"\nOverall Mask Coverage:")
print(f"  Mean: {all_coverage.mean():.2f}%")
print(f"  Std: {all_coverage.std():.2f}%")

## 6. Visualize Original vs Masks

In [ ]:
# Visualize original images vs masks
fig, axes = plt.subplots(3, 6, figsize=(15, 8))

# Show 3 images from each split
for i in range(3):
    # Train
    axes[0, i*2].imshow(X_train[i].squeeze(), cmap='hot')
    axes[0, i*2].set_title(f'Train {i+1} (Original)', fontsize=10)
    axes[0, i*2].axis('off')
    
    axes[0, i*2+1].imshow(masks_train[i].squeeze(), cmap='gray')
    axes[0, i*2+1].set_title(f'Train {i+1} (Mask)', fontsize=10)
    axes[0, i*2+1].axis('off')
    
    # Val
    axes[1, i*2].imshow(X_val[i].squeeze(), cmap='hot')
    axes[1, i*2].set_title(f'Val {i+1} (Original)', fontsize=10)
    axes[1, i*2].axis('off')
    
    axes[1, i*2+1].imshow(masks_val[i].squeeze(), cmap='gray')
    axes[1, i*2+1].set_title(f'Val {i+1} (Mask)', fontsize=10)
    axes[1, i*2+1].axis('off')
    
    # Test
    axes[2, i*2].imshow(X_test[i].squeeze(), cmap='hot')
    axes[2, i*2].set_title(f'Test {i+1} (Original)', fontsize=10)
    axes[2, i*2].axis('off')
    
    axes[2, i*2+1].imshow(masks_test[i].squeeze(), cmap='gray')
    axes[2, i*2+1].set_title(f'Test {i+1} (Mask)', fontsize=10)
    axes[2, i*2+1].axis('off')

plt.suptitle('Galaxy Image Structure - Original vs Spatial Analysis Masks', fontsize=14, y=1.00)
plt.tight_layout()
plt.savefig(PREPROCESSED_DIR / 'masks_visualization.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Visualization saved")

## 7. Save Masks to HDF5

In [ ]:
# Save masks to HDF5
masks_h5_file = PREPROCESSED_DIR / f'masks_{RESOLUTION}.h5'

print(f"Saving masks to {masks_h5_file}...")
with h5py.File(masks_h5_file, 'w') as f:
    f.create_dataset('masks_train', data=masks_train, compression='gzip')
    f.create_dataset('masks_val', data=masks_val, compression='gzip')
    f.create_dataset('masks_test', data=masks_test, compression='gzip')
    
    # Store metadata
    f.attrs['resolution'] = RESOLUTION
    f.attrs['threshold_method'] = THRESHOLD_METHOD
    f.attrs['apply_morphology'] = APPLY_MORPHOLOGY
    f.attrs['morph_size'] = MORPH_SIZE

file_size_mb = masks_h5_file.stat().st_size / 1e6
print(f"✓ Masks saved successfully")
print(f"  File: {masks_h5_file.name}")
print(f"  Size: {file_size_mb:.1f} MB")

# Save coverage statistics
coverage_data = {
    'split': ['train']*len(train_coverage) + ['val']*len(val_coverage) + ['test']*len(test_coverage),
    'mask_coverage_percent': np.concatenate([train_coverage, val_coverage, test_coverage])
}
df_coverage = pd.DataFrame(coverage_data)
df_coverage.to_csv(PREPROCESSED_DIR / f'mask_coverage_{RESOLUTION}.csv', index=False)

print(f"\n✓ Coverage statistics saved: mask_coverage_{RESOLUTION}.csv")

## 8. Final Summary

In [ ]:
print("\n" + "="*70)
print("SPATIAL ANALYSIS MASKS SUMMARY")
print("="*70)

print(f"\n🎯 Masking Configuration:")
print(f"  Method: {THRESHOLD_METHOD} thresholding")
print(f"  Morphology: {'Yes' if APPLY_MORPHOLOGY else 'No'} (size={MORPH_SIZE})")
print(f"  Resolution: {RESOLUTION}x{RESOLUTION}")

print(f"\n📊 Mask Statistics:")
print(f"  Train masks: {masks_train.shape}")
print(f"  Val masks: {masks_val.shape}")
print(f"  Test masks: {masks_test.shape}")

print(f"\n💾 Output Files:")
print(f"  Masks HDF5: {masks_h5_file.name} ({file_size_mb:.1f} MB)")
print(f"  Coverage CSV: mask_coverage_{RESOLUTION}.csv")
print(f"  Visualization: masks_visualization.png")

print(f"\n📈 Coverage Summary:")
print(f"  Train: {train_coverage.mean():.2f}% ± {train_coverage.std():.2f}%")
print(f"  Val: {val_coverage.mean():.2f}% ± {val_coverage.std():.2f}%")
print(f"  Test: {test_coverage.mean():.2f}% ± {test_coverage.std():.2f}%")
print(f"  Overall: {all_coverage.mean():.2f}% ± {all_coverage.std():.2f}%")

print(f"\n✅ Mask generation complete!")
print(f"   Ready for model training with localization targets.")
print("="*70)

## Expected output

The notebook creates spatial analysis masks, saves them to an HDF5 file, writes a visualization image, and prints mask coverage statistics.